### Calculating everything needed as tarHMM input (real 600x600 ROI runs)
- Use SAM3 tracks on the 600x600 crops
- use AnalysisEnv

In [2]:
import pickle
import numpy as np

import zipfile
from io import BytesIO
import tifffile

import os

from pathlib import Path
import yaml

os.chdir("/gladstone/engelhardt/lab/jutran/lci/MarsonImagingPipeline")
from scripts.utils.StatUtils import *
from scripts.utils.CellTypingUtils import *

### Load tracks

In [3]:
sam3_run_dir = "/gladstone/engelhardt/lab/MarsonLabIncucyteData/SnakemakeRuns/Arrietty_Crops_85t_150xy/output/"
cell_typing_run_dir = "/gladstone/engelhardt/lab/MarsonLabIncucyteData/SnakemakeRuns/CellTyping_85t_600xy_full/output/"

well_ids = ["B3"]

In [4]:
sam3_tracks_per_well = {}
t_cell_tracks_per_well = {}
cancer_tracks_per_well = {}

for well in well_ids:
    sam3_tracks = tifffile.imread(os.path.join(sam3_run_dir, well, "full_tracks.tiff"))
    sam3_tracks_per_well[well] = sam3_tracks

    cell_type_dict = pickle.load(open(os.path.join(cell_typing_run_dir, well, "cell_type_dict.pkl"), "rb"))
    t_cell_tracks_per_well[well] = filter_tracks("t_cell", sam3_tracks, cell_type_dict)
    cancer_tracks_per_well[well] = filter_tracks("cancer", sam3_tracks, cell_type_dict)

In [5]:
sam3_tracks_per_well["B3"].shape, t_cell_tracks_per_well["B3"].shape, cancer_tracks_per_well["B3"].shape

((250, 600, 600), (250, 600, 600), (250, 600, 600))

### Calculate input metadata

In [6]:
# obtain max num_cells first
max_num_cells = 0
for well in well_ids:
    t_cell_tracks = t_cell_tracks_per_well[well]
    T = t_cell_tracks.shape[0]
    all_cell_ids = np.unique(t_cell_tracks[t_cell_tracks > 0])
    num_cells = len(all_cell_ids)
    print(f"well={well}, T={T}, num_cells={num_cells}")
    if num_cells > max_num_cells:
        max_num_cells = num_cells
    print(f"max_num_cells so far: {max_num_cells}")

well=B3, T=250, num_cells=1722
max_num_cells so far: 1722


In [7]:
# calculate actual statistics
data = {}

# Initialize all masks
active_mask = np.zeros([50,0], dtype=bool)
is_division_mask = np.zeros([50,0], dtype=bool)
is_new_root_mask = np.zeros([50,0], dtype=bool)
parent_indices = np.zeros([50,0], dtype=np.int32)

active_mask_list = []
is_division_mask_list = []
is_new_root_mask_list = []
parent_indices_list = []

for well in well_ids:
    t_cell_tracks = t_cell_tracks_per_well[well]

    T = t_cell_tracks.shape[0]
    all_cell_ids = np.unique(t_cell_tracks[t_cell_tracks > 0])
    all_cell_ids.sort()
    num_cells = len(all_cell_ids)
    id_to_col = {int(cid): i for i, cid in enumerate(all_cell_ids)}

    print(f"well={well}, T={T}, num_cells={num_cells}")

    crop_active_mask = np.zeros((T, max_num_cells), dtype=bool)
    crop_is_division_mask = np.zeros((T, max_num_cells), dtype=bool)
    crop_is_new_root_mask = np.zeros((T, max_num_cells), dtype=bool)
    crop_parent_indices = np.zeros((T, max_num_cells), dtype=np.int32)

    # Make active_mask from t_cell_tracks
    for t in range(T):
        t_cell_frame = t_cell_tracks[t]
        frame_ids = np.unique(t_cell_frame[t_cell_frame > 0])
        for cid in frame_ids:
            crop_active_mask[t, id_to_col[int(cid)]] = True

    # 4. Build is_new_root_mask, parent_indices (keep is_division_mask as all False)
    for cid, col in id_to_col.items():
        active_frames = np.where(crop_active_mask[:, col])[0]
        if len(active_frames) == 0:
            print(f"Warning: Cell ID {cid} (col {col}) is never active in active_mask. Skipping.")
            continue

        first_frame = active_frames[0]

        # This cell is a root (appeared spontaneously or is the initial cell)
        crop_is_new_root_mask[first_frame, col] = True
        crop_parent_indices[first_frame, col] = col  # self

        # For all subsequent active frames, parent = self (cell continues)
        for t in active_frames[1:]:
            crop_parent_indices[t, col] = col

    # Append well masks to overall masks
    active_mask_list.append(crop_active_mask)
    is_division_mask_list.append(crop_is_division_mask)
    is_new_root_mask_list.append(crop_is_new_root_mask)
    parent_indices_list.append(crop_parent_indices)

well=B3, T=250, num_cells=1722


In [9]:
active_mask = np.array(active_mask_list)
is_division_mask = np.array(is_division_mask_list)
is_new_root_mask = np.array(is_new_root_mask_list)
parent_indices = np.array(parent_indices_list)

print(f"active_mask shape:       {active_mask.shape}")
print(f"is_division_mask shape:  {is_division_mask.shape}")
print(f"is_new_root_mask shape:  {is_new_root_mask.shape}")
print(f"parent_indices shape:    {parent_indices.shape}")

active_mask shape:       (1, 250, 1722)
is_division_mask shape:  (1, 250, 1722)
is_new_root_mask shape:  (1, 250, 1722)
parent_indices shape:    (1, 250, 1722)


In [10]:
data['active_mask'] = active_mask
data['is_division_mask'] = is_division_mask
data['is_new_root_mask'] = is_new_root_mask
data['parent_indices'] = parent_indices

### Calculate emissions feature values

In [11]:
# calculate t cell velocities
t_cell_velocities_per_frame = {well: compute_cell_velocities_per_frame_dict(t_cell_tracks_per_well[well], unit_per_frame=1) for well in t_cell_tracks_per_well.keys()}


Computing cell velocities: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 249/249 [00:14<00:00, 17.45it/s]


In [12]:
# calculate type-specific interactions

cancer_type_specific_contacts_per_frame, t_cell_type_specific_contacts_per_frame = {}, {}
cancer_type_specific_neighbors_per_frame, t_cell_type_specific_neighbors_per_frame = {}, {}

for well in sam3_tracks_per_well.keys():
    t_cell_tracks = t_cell_tracks_per_well[well]
    cancer_tracks = cancer_tracks_per_well[well]

    well_cancer_contacts_per_frame, well_t_cell_contacts_per_frame = compute_cell_cell_contact_dict(t_cell_tracks, cancer_tracks)
    well_cancer_neighbors_per_frame, well_t_cell_neighbors_per_frame = compute_cell_cell_neighbor_dict(t_cell_tracks, cancer_tracks)

    cancer_type_specific_contacts_per_frame[well] = well_cancer_contacts_per_frame
    t_cell_type_specific_contacts_per_frame[well] = well_t_cell_contacts_per_frame

    cancer_type_specific_neighbors_per_frame[well] = well_cancer_neighbors_per_frame
    t_cell_type_specific_neighbors_per_frame[well] = well_t_cell_neighbors_per_frame

Computing cell-cell contact dataframe...


Processing frame-by-frame contact output: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 250/250 [00:20<00:00, 12.49it/s]


Computing cell-cell neighbor dataframe...


Processing frame-by-frame neighbor output: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 250/250 [00:20<00:00, 12.40it/s]


In [13]:
# calculate type-agnostic interactions

cancer_all_contacts_per_frame, t_cell_all_contacts_per_frame = {}, {}
cancer_all_neighbors_per_frame, t_cell_all_neighbors_per_frame = {}, {}

for well in sam3_tracks_per_well.keys():
    t_cell_tracks = t_cell_tracks_per_well[well]
    cancer_tracks = cancer_tracks_per_well[well]

    well_cancer_contacts_per_frame, well_t_cell_contacts_per_frame = compute_all_cell_cell_contact_dict(t_cell_tracks, cancer_tracks)
    well_cancer_neighbors_per_frame, well_t_cell_neighbors_per_frame = compute_all_cell_cell_neighbor_dict(t_cell_tracks, cancer_tracks)

    cancer_all_contacts_per_frame[well] = well_cancer_contacts_per_frame
    t_cell_all_contacts_per_frame[well] = well_t_cell_contacts_per_frame

    cancer_all_neighbors_per_frame[well] = well_cancer_neighbors_per_frame
    t_cell_all_neighbors_per_frame[well] = well_t_cell_neighbors_per_frame

Computing cell-cell contact dataframe...


Processing contact dict: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 250/250 [00:30<00:00,  8.30it/s]


Computing cell-cell neighbor dataframe...


Processing neighbor dict: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 250/250 [00:30<00:00,  8.31it/s]


In [14]:
# emissions_array = shape: (# wells/batches, num_frames, num_t_cells that is max across all batches, num_emission_features)

emissions_array_list = []

for well in well_ids:
    t_cell_tracks = t_cell_tracks_per_well[well]
    t_cell_velocities = t_cell_velocities_per_frame[well]
    t_cell_type_specific_neighbors = t_cell_type_specific_neighbors_per_frame[well]
    t_cell_all_neighbors = t_cell_all_neighbors_per_frame[well]

    T = t_cell_tracks.shape[0]
    t_cell_ids = np.unique(t_cell_tracks[t_cell_tracks > 0])
    t_cell_ids.sort()
    id_to_column_index = {cell_id: index for index, cell_id in enumerate(t_cell_ids)}
    
    print(f"Well={well}, T={T}, num_cells={len(t_cell_ids)}")

    well_emissions = np.zeros((T, max_num_cells, 3)) # 3 features: velocity, type-specific neighbors, all neighbors

    for t in range(T):
        frame = t_cell_tracks[t]
        for cell_id in np.unique(frame[frame > 0]):
            column_index = id_to_column_index[cell_id]

            if t > 0:
                if cell_id in t_cell_velocities[t]:
                    velocity = t_cell_velocities[t][cell_id]
                    well_emissions[t, column_index, 0] = velocity

            if cell_id in t_cell_all_neighbors[t]:
                all_neighbors = t_cell_all_neighbors[t][cell_id]
                cancer_neighbors_list = t_cell_type_specific_neighbors[t].get(cell_id, [0])
                cancer_neighbors = cancer_neighbors_list[0]

                t_cell_neighbors = all_neighbors - cancer_neighbors

                well_emissions[t, column_index, 1] = cancer_neighbors
                well_emissions[t, column_index, 2] = t_cell_neighbors

    emissions_array_list.append(well_emissions)
    print(f"emissions_array shape after processing {well}: {well_emissions.shape}")

Well=B3, T=250, num_cells=1722
emissions_array shape after processing B3: (250, 1722, 3)


In [16]:
emissions_array = np.array(emissions_array_list)
print(f"Final emissions_array shape: {emissions_array.shape}")

Final emissions_array shape: (1, 250, 1722, 3)


### Generate well-specific array col idx mapping

In [17]:
all_well_cell_ids = []

for well in well_ids:
    t_cell_tracks = t_cell_tracks_per_well[well]
    t_cell_ids = np.unique(t_cell_tracks[t_cell_tracks > 0])
    t_cell_ids.sort()

    filler_cell_ids = [f"{well}_filler_{i}" for i in range(max_num_cells - len(t_cell_ids))]

    well_t_cell_ids = [f"{well}_{cell_id}" for cell_id in t_cell_ids]

    # add filler bc need to have consistent cell numbers
    well_t_cell_ids.extend(filler_cell_ids)

    all_well_cell_ids.extend(well_t_cell_ids)

id_to_col = {cid: i for i, cid in enumerate(all_well_cell_ids)}

In [18]:
len(id_to_col.keys())

1722

In [19]:
output_dir = "/gladstone/engelhardt/lab/jutran/lci/treeHMM/notebooks/data/example_600x600_well/"

with open(output_dir + "batched_B3_data.pkl", "wb") as f:
    pickle.dump(data, f)

with open(output_dir + "batched_B3_well_and_cell_id_to_col.pkl", "wb") as f:
    pickle.dump(id_to_col, f)

np.save(output_dir + "batched_B3_emissions_array.npy", emissions_array)